# 02 — Preprocessing: 10-min grid construction

**Objective**: Build a uniform 10-minute grid from mixed-frequency clean CSVs (Phase 2 of the work plan).

### Resampling decisions (documented for thesis)

| Variable type | 5 min -> 10 min | 15 min -> 10 min | 60 min -> 10 min |
|---|---|---|---|
| HEIGHT_m, TEMP_C (instantaneous) | Mean of 2 consecutive | Linear interpolation | — |
| PRECIP_mm — STM (cumulative) | Cumsum -> interpolate -> diff | Cumsum -> interpolate -> diff | — |
| PRECIP_mm — AEMET (hourly) | — | — | Template-based (STM01/STM02 pattern) or conservative cumulative |


## 1. Environment setup

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

REPO = Path.cwd().resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.preprocessing.resample import (
    build_10min_grid, load_all,
    HYDRO, METEO, AEMET,
)

CLEAN = REPO / "data" / "clean"
PROCESSED = REPO / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
print(f"Repository root: {REPO}")


ModuleNotFoundError: No module named 'scripts'

## 2. Load raw data (pre-resampling)

In [ ]:
data_raw = load_all(CLEAN)
print("Raw data summary:")
for code in HYDRO + METEO + AEMET:
    df = data_raw[code]
    diffs = df.index.to_series().diff().dropna()
    mf = diffs.mode().iloc[0] if len(diffs.mode()) > 0 else pd.NaT
    print(f"  {code:6s}  rows={len(df):,}  mode_freq={mf}  cols={list(df.columns)}")


## 3. Build 10-min grid

**Window**: 2014-09-26 (STM02 start) -> 2025-07-22 (end of validated data).
Gaps <= 3 h are interpolated; larger gaps are left as NaN.


In [ ]:
GRID_PATH = PROCESSED / "grid_10min.parquet"
if GRID_PATH.exists():
    print(f"Loading cached grid ...")
    grid = pd.read_parquet(GRID_PATH)
else:
    grid, gap_info = build_10min_grid(start="2014-09-26", end="2025-07-22", clean_dir=CLEAN)
    grid.to_parquet(GRID_PATH)
print(f"Shape: {grid.shape[0]:,} x {grid.shape[1]}")
print(f"Period: {grid.index.min()} -> {grid.index.max()}")
print(f"Memory: {grid.memory_usage(deep=True).sum()/1e6:.1f} MB")


## 4. Null-rate analysis

In [ ]:
meas_cols = [c for c in grid.columns if not c.endswith("_QUALITY")]
null_pct = grid[meas_cols].isnull().mean().mul(100).round(1)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(null_pct.index, null_pct.values, color=["steelblue"]*6 + ["darkorange"]*4 + ["forestgreen"]*3)
ax.set_xlabel("Null %")
ax.set_title("Missing data per station/variable (10-min grid)")
ax.invert_yaxis()
for bar, val in zip(bars, null_pct.values):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, f"{val}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()
print(null_pct.to_string())


## 5. Verify resampling quality

Compare raw vs. resampled STM08 during a flood event.


In [ ]:
zoom_start, zoom_end = "2016-12-15", "2016-12-22"
raw = data_raw["STM08"]["HEIGHT_m"].loc[zoom_start:zoom_end].dropna()
resampled = grid["STM08_HEIGHT_m"].loc[zoom_start:zoom_end]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(raw.index, raw.values, "o", ms=3, alpha=0.6, label="Raw (15-min)", color="gray")
ax.plot(resampled.index, resampled.values, "-", lw=1.2, label="10-min grid", color="tab:blue")
ax.set_ylabel("HEIGHT_m")
ax.set_title(f"STM08 — raw vs 10-min grid ({zoom_start} to {zoom_end})")
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout()
plt.show()


## 6. Precipitation resampling verification

Hourly AEMET totals must equal the sum of their 10-min disaggregated values.


In [ ]:
code_check = "B013X"
w = "2020-01-15", "2020-01-25"
raw_aemet = data_raw[code_check]["PRECIP_mm"].loc[w[0]:w[1]].dropna()
resampled = grid[f"{code_check}_PRECIP_mm"].loc[w[0]:w[1]]
hourly_sum = resampled.resample("1h", closed="right", label="right").sum()
common = raw_aemet.index.intersection(hourly_sum.index)
if len(common) > 0:
    diff = (hourly_sum.loc[common] - raw_aemet.loc[common]).abs()
    print(f"Mass conservation ({code_check}): max_err={diff.max():.6f} mm  mean_err={diff.mean():.6f} mm  n={len(common)}")


## 7. Temperature resampling check

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
z = "2020-07-15", "2020-07-22"
for ax, code in zip(axes, ["STM01", "STM02"]):
    raw_t = data_raw[code]["TEMP_C"].loc[z[0]:z[1]].dropna()
    res_t = grid[f"{code}_TEMP_C"].loc[z[0]:z[1]]
    ax.plot(raw_t.index, raw_t.values, "o", ms=3, alpha=0.5, label=f"Raw", color="gray")
    ax.plot(res_t.index, res_t.values, "-", lw=1, label="10-min", color="tab:red")
    ax.set_ylabel("TEMP_C"); ax.legend(fontsize=9); ax.set_title(code)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.tight_layout(); plt.show()


## 8. Final grid inventory

In [ ]:
mc = [c for c in grid.columns if not c.endswith("_QUALITY")]
print(f"Grid: {grid.shape[0]:,} rows x {grid.shape[1]} cols, 10-min, { (grid.index.max()-grid.index.min()).days/365.25:.1f} years")
for col in mc:
    s = grid[col]; nn = s.notna().sum()
    print(f"  {col:25s}  non-null={nn:>8,}  null={100*(1-nn/len(s)):5.1f}%  mean={s.mean():.3f}")
qc = [c for c in grid.columns if c.endswith("_QUALITY")]
for col in qc:
    s = grid[col].dropna()
    if len(s) > 0:
        print(f"  {col:25s}  non-null={len(s):>8,}  values={s.value_counts().to_dict()}")
    else:
        print(f"  {col:25s}  non-null=        0  (Excel data)")
